# Pratica 10 — Correlazione e regressione lineare

**Corso:** Statistica Applicata con Python · **Lezione 10** · **Tempo stimato:** 75-100 minuti
**Notebook di lezione:** `notebooks_clase/clase10_correlacion_regresion_lineal_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi con simulazione usano tolleranze basate sull'errore standard, quindi un risultato corretto non dovrebbe fallire per caso.
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Gli esempi piccoli e le simulazioni sono *esempi illustrativi* sintetici; gli esercizi da 3 a 5 usano i dati di California housing (CSV locale, download o alternativa sintetica, come nella lezione).

---
## Cella 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

In [ ]:
def generar_california_sintetico(n=5000, seed=42):
    """Genera un dataset SINTETICO con le stesse colonne di California housing (alternativa se non c'è il CSV né il download).
    Le relazioni sono inventate: serve per esercitarsi, non per trarre conclusioni sulla California."""
    g = np.random.default_rng(seed)
    MedInc = np.clip(g.lognormal(1.25, 0.5, n), 0.5, 15)
    HouseAge = g.integers(1, 53, n).astype(float)
    AveRooms = np.clip(3.0 + 0.35 * MedInc + g.normal(0, 0.7, n), 2, 9)
    AveBedrms = np.clip(0.2 * AveRooms + g.normal(0, 0.05, n), 0.6, 2)
    Population = np.round(g.lognormal(7.0, 0.6, n))
    AveOccup = np.clip(g.lognormal(1.05, 0.25, n), 1.2, 5.5)
    Latitude = np.round(g.uniform(32.5, 42.0, n), 2)
    Longitude = np.round(g.uniform(-124.3, -114.5, n), 2)
    valor = (0.5 + 0.42 * MedInc + 0.012 * HouseAge - 0.15 * AveRooms - 0.3 * AveOccup
             + 0.05 * (37 - Latitude) + g.normal(0, 0.6, n))
    MedHouseVal = np.clip(valor, 0.15, 5.00001)          # censura a 5,00001, come l'originale
    return pd.DataFrame({"MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms, "AveBedrms": AveBedrms,
                         "Population": Population, "AveOccup": AveOccup, "Latitude": Latitude,
                         "Longitude": Longitude, "MedHouseVal": MedHouseVal})


def cargar_california():
    """Restituisce (DataFrame, origine): CSV locale, altrimenti download di scikit-learn, altrimenti dati sintetici."""
    for ruta in ("../datos/california_housing.csv", "datos/california_housing.csv", "california_housing.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV locale"
    try:
        from sklearn.datasets import fetch_california_housing
        datos = fetch_california_housing(as_frame=True).frame
        return datos.sample(n=5000, random_state=42).reset_index(drop=True), "scikit-learn"
    except Exception:
        return generar_california_sintetico(), "sintetico"


df_bruto, origen = cargar_california()
print("Origine dei dati:", origen, "|", df_bruto.shape[0], "righe")

In [ ]:
# Preparazione: togliamo distretti con valori medi assurdi e creiamo la variabile di zona
df = df_bruto[(df_bruto["AveRooms"] < 10) & (df_bruto["AveOccup"] < 6)].copy()
df["zona"] = np.where(df["Latitude"] >= 36, "Nord", "Sud")
print(f"Distretti usati: {len(df)} su {len(df_bruto)}")

In [ ]:
def r_de(x, y):
    """Correlazione di Pearson tra due vettori."""
    return np.corrcoef(x, y)[0, 1]

## Esercizio 1 — Pearson e Spearman a mano

Sette punti con una relazione **crescente ma curva** (*esempio illustrativo*): `y = x²`. Calcola, senza usare `np.corrcoef` né `stats.spearmanr`:

- `cov_xy`: covarianza campionaria (`ddof=1`),
- `r_pearson`: covarianza divisa per il prodotto delle deviazioni standard (con `ddof=1`),
- `rho_spearman`: correlazione di Pearson calcolata sui **ranghi** (`stats.rankdata`).

Domanda su cui riflettere: perché `rho_spearman` vale esattamente 1 e `r_pearson` no?

In [ ]:
x1 = np.arange(1, 8, dtype=float)
y1 = x1 ** 2

cov_xy = None          # TODO: somma dei prodotti degli scarti / (n - 1)
r_pearson = None       # TODO
rho_spearman = None    # TODO: Pearson dei ranghi di x1 e di y1

# Controllo automatico
assert all(v is not None for v in (cov_xy, r_pearson, rho_spearman)), "Completa le righe contrassegnate con TODO"
assert np.isclose(cov_xy, np.cov(x1, y1)[0, 1]), "la covarianza usa ddof=1: divide per n - 1"
assert np.isclose(r_pearson, np.corrcoef(x1, y1)[0, 1]), "r = cov / (sx · sy), con ddof=1 nelle tre quantità"
assert np.isclose(rho_spearman, 1.0), "i ranghi di x1 e di y1 coincidono: la correlazione dei ranghi vale 1"
assert 0.95 < r_pearson < 1.0, "Pearson è alto ma minore di 1: la relazione è curva"
print(f"Covarianza = {cov_xy:.2f}   Pearson = {r_pearson:.4f}   Spearman = {rho_spearman:.1f}")
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Minimi quadrati a mano

Cinque coppie (*esempio illustrativo*): `x = 2, 4, 6, 8, 10` (investimento in pubblicità, migliaia di €) e `y = 3, 7, 8, 12, 15` (vendite, migliaia di €). Calcola con le formule (senza `polyfit` né `statsmodels`):

- `b1` (pendenza) e `b0` (intercetta),
- `pred_7`: previsione per `x = 7`,
- `ssr`: somma dei quadrati dei residui, e `r2 = 1 - ssr / sst`, con `sst` la somma dei quadrati totale di `y` rispetto alla sua media.

In [ ]:
x2 = np.array([2, 4, 6, 8, 10], dtype=float)
y2 = np.array([3, 7, 8, 12, 15], dtype=float)

b1 = None        # TODO: Sxy / Sxx
b0 = None        # TODO: media di y meno b1 per la media di x
pred_7 = None    # TODO
ssr = None       # TODO: somma di (y - previsto)²
sst = None       # TODO: somma di (y - media di y)²
r2 = None        # TODO

# Controllo automatico
assert all(v is not None for v in (b1, b0, pred_7, ssr, sst, r2)), "Completa le righe contrassegnate con TODO"
assert np.isclose(b1, 1.45), "b1 = Sxy / Sxx = 58 / 40"
assert np.isclose(b0, 0.3), "b0 = 9 - 1,45 · 6"
assert np.isclose(pred_7, 10.45), "pred_7 = b0 + b1 · 7"
assert np.isclose(ssr, 1.9) and np.isclose(sst, 86), "rivedi i residui e la somma totale"
assert np.isclose(r2, np.corrcoef(x2, y2)[0, 1] ** 2), "con una sola variabile, R² = r²"
print(f"b1 = {b1:.2f}   b0 = {b0:.2f}   previsione in x=7 = {pred_7:.2f}   R² = {r2:.4f}")
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Regressione multipla con `statsmodels`

Con i dati della California (`df`, già preparato nel setup), adatta `MedHouseVal ~ MedInc + HouseAge` con `smf.ols(...).fit()` ed estrai:

- `modelo`: il risultato dell'adattamento,
- `coef_ingreso`: coefficiente di `MedInc`, e `ic_ingreso`: il suo intervallo di confidenza al 95 % come tupla `(inferiore, superiore)`,
- `p_ingreso`: il suo p-value,
- `efecto_10_anios`: variazione attesa del valore per 10 anni in più di età delle abitazioni, **a parità di reddito**,
- `r2` e `r2_aj`: R² e R² corretto.

In [ ]:
modelo = None            # TODO: smf.ols("MedHouseVal ~ MedInc + HouseAge", data=df).fit()
coef_ingreso = None      # TODO
ic_ingreso = None        # TODO: tupla (inferiore, superiore) da conf_int()
p_ingreso = None         # TODO
efecto_10_anios = None   # TODO
r2 = None                # TODO
r2_aj = None             # TODO

# Controllo automatico (contro un calcolo indipendente con algebra lineale)
assert all(v is not None for v in (modelo, coef_ingreso, ic_ingreso, p_ingreso, efecto_10_anios, r2, r2_aj)), "Completa le righe contrassegnate con TODO"
X_ref = np.column_stack([np.ones(len(df)), df["MedInc"], df["HouseAge"]])
y_ref = df["MedHouseVal"].values
beta = np.linalg.lstsq(X_ref, y_ref, rcond=None)[0]
res_ref = y_ref - X_ref @ beta
gl_ref = len(df) - 3
s2 = (res_ref ** 2).sum() / gl_ref
ee_ref = np.sqrt(np.diag(s2 * np.linalg.inv(X_ref.T @ X_ref)))
t_ref = stats.t.ppf(0.975, gl_ref)
assert np.isclose(coef_ingreso, beta[1]), "il coefficiente di MedInc si trova in modelo.params"
assert np.allclose(ic_ingreso, (beta[1] - t_ref * ee_ref[1], beta[1] + t_ref * ee_ref[1]), atol=1e-6), "usa modelo.conf_int() e prendi la riga di MedInc"
assert ic_ingreso[0] < coef_ingreso < ic_ingreso[1], "il coefficiente deve cadere dentro il suo intervallo"
assert 0 <= p_ingreso < 0.05, "rivedi modelo.pvalues"
assert np.isclose(efecto_10_anios, 10 * beta[2]), "l'effetto di 10 anni è 10 volte il coefficiente di HouseAge"
assert np.isclose(r2, 1 - (res_ref ** 2).sum() / ((y_ref - y_ref.mean()) ** 2).sum()), "R² = 1 - SSR/SST; si trova in modelo.rsquared"
assert r2_aj < r2, "l'R² corretto è sempre minore di R² (con più di una variabile)"
print(f"MedInc: {coef_ingreso:.4f}   IC 95 %: ({ic_ingreso[0]:.4f}, {ic_ingreso[1]:.4f})   R² = {r2:.4f}   R² corretto = {r2_aj:.4f}")
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Previsione con due intervalli

Con il modello semplice `m_ej = smf.ols("MedHouseVal ~ MedInc", data=df).fit()`, completa:

- `intervalos(modelo, medinc)`: restituisce un dizionario con `"pred"` (valore previsto), `"ic_media"` (tupla dell'IC 95 % della media) e `"int_pred"` (tupla dell'intervallo di previsione 95 %). Suggerimento: `modelo.get_prediction(...).summary_frame(alpha=0.05)`.
- `en_rango(medinc)`: `True` se `medinc` è dentro il range osservato di `df["MedInc"]`.

In [ ]:
m_ej = smf.ols("MedHouseVal ~ MedInc", data=df).fit()

def intervalos(modelo, medinc):
    """Previsione, IC 95 % della media e intervallo di previsione 95 % per un valore di MedInc."""
    # TODO: usa get_prediction su un DataFrame con la colonna MedInc
    return None

def en_rango(medinc):
    """True se medinc è dentro il range osservato di MedInc."""
    # TODO
    return None

# Controllo automatico
r4 = intervalos(m_ej, 4.0)
assert r4 is not None and en_rango(4.0) is not None, "Completa le funzioni (TODO)"
b0_, b1_ = m_ej.params["Intercept"], m_ej.params["MedInc"]
assert np.isclose(r4["pred"], b0_ + b1_ * 4.0), "la previsione è b0 + b1 · x"
assert r4["ic_media"][0] < r4["pred"] < r4["ic_media"][1], "la previsione deve stare dentro l'IC della media"
assert r4["int_pred"][0] < r4["ic_media"][0] and r4["int_pred"][1] > r4["ic_media"][1], "l'intervallo di previsione contiene l'IC della media ed è più ampio"
ancho_media = r4["ic_media"][1] - r4["ic_media"][0]
ancho_pred = r4["int_pred"][1] - r4["int_pred"][0]
assert ancho_pred > 10 * ancho_media, "con migliaia di dati l'IC della media è molto più stretto di quello di previsione"
assert en_rango(4.0) is True and en_rango(30.0) is False and en_rango(0.0) is False, "confronta con il minimo e il massimo di df['MedInc']"
print(f"Ampiezze: IC della media = {ancho_media:.3f}   intervallo di previsione = {ancho_pred:.3f}")
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Diagnostica dei residui

Con `m_ej` (esercizio 4), calcola:

- `techo`: maschera booleana dei distretti con `MedHouseVal >= 5` (valori censurati), `n_techo`: quanti sono,
- `res_techo` e `res_resto`: residuo medio dei censurati e degli altri,
- `cook`: distanze di Cook (`m_ej.get_influence().cooks_distance[0]`) e `pos_cook_max`: posizione della maggiore,
- `corr_res_ajust`: correlazione tra residui e valori adattati.

Domanda su cui riflettere: perché i residui dei distretti censurati sono in media **positivi**?

In [ ]:
residuos = m_ej.resid.values
ajustados = m_ej.fittedvalues.values

techo = None            # TODO: maschera booleana (df["MedHouseVal"] >= 5).values
n_techo = None          # TODO
res_techo = None        # TODO
res_resto = None        # TODO
cook = None             # TODO
pos_cook_max = None     # TODO
corr_res_ajust = None   # TODO

# Controllo automatico
assert all(v is not None for v in (techo, n_techo, res_techo, res_resto, cook, pos_cook_max, corr_res_ajust)), "Completa le righe contrassegnate con TODO"
assert n_techo == int((df["MedHouseVal"] >= 5).sum()) and n_techo > 0, "conta i distretti con MedHouseVal >= 5"
assert np.isclose(res_techo, residuos[techo].mean()) and np.isclose(res_resto, residuos[~techo].mean()), "usa la maschera techo e la sua negazione"
assert res_techo > 0 > res_resto, "i censurati hanno residuo positivo: il loro valore reale (non troncato) sarebbe ancora maggiore"
assert len(cook) == len(df) and pos_cook_max == int(np.argmax(cook)), "pos_cook_max è la posizione del massimo di cook"
assert abs(corr_res_ajust) < 1e-8, "in OLS con intercetta, i residui non sono correlati con i valori adattati"
print(f"Censurati: {n_techo}   residuo medio censurati / altri: {res_techo:+.2f} / {res_resto:+.2f}   Cook max.: {cook.max():.3f}")
print("Esercizio 5 corretto ✔")

---
## Sfida finale — La confusione che inganna

Una simulazione (*esempio illustrativo*): la variabile `z` influenza `x` e `y`, e `x` ha un effetto **vero** di 0,3 su `y`. Un'analisi ingenua che ignori `z` sovrastima quell'effetto.

Completa:

- `coef_crudo`, `ee_crudo`: coefficiente ed errore standard di `x` in `y ~ x`,
- `coef_ajustado`, `ee_ajustado`: lo stesso in `y ~ x + z`,
- `sesgo`: `coef_crudo - coef_ajustado`.

Poi scrivi (nella cella di conclusioni) cosa servirebbe per attribuire a `x` un effetto **causale**.

In [ ]:
g = np.random.default_rng(11)
n_r = 3000
z_r = g.normal(0, 1, n_r)
x_r = 0.8 * z_r + g.normal(0, 1, n_r)
y_r = 0.3 * x_r + 1.0 * z_r + g.normal(0, 1, n_r)
datos_r = pd.DataFrame({"x": x_r, "y": y_r, "z": z_r})

coef_crudo = None       # TODO
ee_crudo = None         # TODO
coef_ajustado = None    # TODO
ee_ajustado = None      # TODO
sesgo = None            # TODO

# Controllo automatico (tolleranze basate sull'errore standard)
assert all(v is not None for v in (coef_crudo, ee_crudo, coef_ajustado, ee_ajustado, sesgo)), "Completa le righe contrassegnate con TODO"
assert abs(coef_ajustado - 0.3) < 4 * ee_ajustado, "il modello con z deve recuperare l'effetto vero 0,3 (entro 4 errori standard)"
assert coef_crudo - 0.3 > 4 * ee_crudo, "il modello senza z sovrastima l'effetto (più di 4 errori standard)"
assert np.isclose(sesgo, coef_crudo - coef_ajustado) and sesgo > 0.3, "il bias è grezzo - corretto e qui supera 0,3"
print(f"Effetto vero: 0,3   grezzo: {coef_crudo:.3f} ± {ee_crudo:.3f}   corretto: {coef_ajustado:.3f} ± {ee_ajustado:.3f}")
print("bias:", round(sesgo, 3))
print("Sfida finale corretta ✔")

**Le tue conclusioni (scrivi 3-4 frasi):** cosa servirebbe per poter dire che `x` *causa* `y`? Quando è utile un modello di regressione anche se non permette di parlare di causalità? Quali limiti dei dati della California (censura a 5,00001, dati del 1990, distretti e non singole abitazioni) dovresti citare in un rapporto?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
x1 = np.arange(1, 8, dtype=float)
y1 = x1 ** 2

cov_xy = ((x1 - x1.mean()) * (y1 - y1.mean())).sum() / (len(x1) - 1)
r_pearson = cov_xy / (x1.std(ddof=1) * y1.std(ddof=1))
rho_spearman = np.corrcoef(stats.rankdata(x1), stats.rankdata(y1))[0, 1]

assert all(v is not None for v in (cov_xy, r_pearson, rho_spearman)), "Completa le righe contrassegnate con TODO"
assert np.isclose(cov_xy, np.cov(x1, y1)[0, 1]), "la covarianza usa ddof=1: divide per n - 1"
assert np.isclose(r_pearson, np.corrcoef(x1, y1)[0, 1]), "r = cov / (sx · sy), con ddof=1 nelle tre quantità"
assert np.isclose(rho_spearman, 1.0), "i ranghi di x1 e di y1 coincidono: la correlazione dei ranghi vale 1"
assert 0.95 < r_pearson < 1.0, "Pearson è alto ma minore di 1: la relazione è curva"
print(f"Covarianza = {cov_xy:.2f}   Pearson = {r_pearson:.4f}   Spearman = {rho_spearman:.1f}")
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
x2 = np.array([2, 4, 6, 8, 10], dtype=float)
y2 = np.array([3, 7, 8, 12, 15], dtype=float)

b1 = ((x2 - x2.mean()) * (y2 - y2.mean())).sum() / ((x2 - x2.mean()) ** 2).sum()
b0 = y2.mean() - b1 * x2.mean()
pred_7 = b0 + b1 * 7
ssr = ((y2 - (b0 + b1 * x2)) ** 2).sum()
sst = ((y2 - y2.mean()) ** 2).sum()
r2 = 1 - ssr / sst

assert all(v is not None for v in (b1, b0, pred_7, ssr, sst, r2)), "Completa le righe contrassegnate con TODO"
assert np.isclose(b1, 1.45), "b1 = Sxy / Sxx = 58 / 40"
assert np.isclose(b0, 0.3), "b0 = 9 - 1,45 · 6"
assert np.isclose(pred_7, 10.45), "pred_7 = b0 + b1 · 7"
assert np.isclose(ssr, 1.9) and np.isclose(sst, 86), "rivedi i residui e la somma totale"
assert np.isclose(r2, np.corrcoef(x2, y2)[0, 1] ** 2), "con una sola variabile, R² = r²"
print(f"b1 = {b1:.2f}   b0 = {b0:.2f}   previsione in x=7 = {pred_7:.2f}   R² = {r2:.4f}")
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
modelo = smf.ols("MedHouseVal ~ MedInc + HouseAge", data=df).fit()
coef_ingreso = modelo.params["MedInc"]
ic_ingreso = tuple(modelo.conf_int().loc["MedInc"])
p_ingreso = modelo.pvalues["MedInc"]
efecto_10_anios = 10 * modelo.params["HouseAge"]
r2 = modelo.rsquared
r2_aj = modelo.rsquared_adj

assert all(v is not None for v in (modelo, coef_ingreso, ic_ingreso, p_ingreso, efecto_10_anios, r2, r2_aj)), "Completa le righe contrassegnate con TODO"
X_ref = np.column_stack([np.ones(len(df)), df["MedInc"], df["HouseAge"]])
y_ref = df["MedHouseVal"].values
beta = np.linalg.lstsq(X_ref, y_ref, rcond=None)[0]
res_ref = y_ref - X_ref @ beta
gl_ref = len(df) - 3
s2 = (res_ref ** 2).sum() / gl_ref
ee_ref = np.sqrt(np.diag(s2 * np.linalg.inv(X_ref.T @ X_ref)))
t_ref = stats.t.ppf(0.975, gl_ref)
assert np.isclose(coef_ingreso, beta[1]), "il coefficiente di MedInc si trova in modelo.params"
assert np.allclose(ic_ingreso, (beta[1] - t_ref * ee_ref[1], beta[1] + t_ref * ee_ref[1]), atol=1e-6), "usa modelo.conf_int() e prendi la riga di MedInc"
assert ic_ingreso[0] < coef_ingreso < ic_ingreso[1], "il coefficiente deve cadere dentro il suo intervallo"
assert 0 <= p_ingreso < 0.05, "rivedi modelo.pvalues"
assert np.isclose(efecto_10_anios, 10 * beta[2]), "l'effetto di 10 anni è 10 volte il coefficiente di HouseAge"
assert np.isclose(r2, 1 - (res_ref ** 2).sum() / ((y_ref - y_ref.mean()) ** 2).sum()), "R² = 1 - SSR/SST; si trova in modelo.rsquared"
assert r2_aj < r2, "l'R² corretto è sempre minore di R² (con più di una variabile)"
print(f"MedInc: {coef_ingreso:.4f}   IC 95 %: ({ic_ingreso[0]:.4f}, {ic_ingreso[1]:.4f})   R² = {r2:.4f}   R² corretto = {r2_aj:.4f}")
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
m_ej = smf.ols("MedHouseVal ~ MedInc", data=df).fit()

def intervalos(modelo, medinc):
    """Previsione, IC 95 % della media e intervallo di previsione 95 % per un valore di MedInc."""
    fila = modelo.get_prediction(pd.DataFrame({"MedInc": [medinc]})).summary_frame(alpha=0.05).iloc[0]
    return {"pred": fila["mean"],
            "ic_media": (fila["mean_ci_lower"], fila["mean_ci_upper"]),
            "int_pred": (fila["obs_ci_lower"], fila["obs_ci_upper"])}

def en_rango(medinc):
    """True se medinc è dentro il range osservato di MedInc."""
    return bool(df["MedInc"].min() <= medinc <= df["MedInc"].max())

r4 = intervalos(m_ej, 4.0)
assert r4 is not None and en_rango(4.0) is not None, "Completa le funzioni (TODO)"
b0_, b1_ = m_ej.params["Intercept"], m_ej.params["MedInc"]
assert np.isclose(r4["pred"], b0_ + b1_ * 4.0), "la previsione è b0 + b1 · x"
assert r4["ic_media"][0] < r4["pred"] < r4["ic_media"][1], "la previsione deve stare dentro l'IC della media"
assert r4["int_pred"][0] < r4["ic_media"][0] and r4["int_pred"][1] > r4["ic_media"][1], "l'intervallo di previsione contiene l'IC della media ed è più ampio"
ancho_media = r4["ic_media"][1] - r4["ic_media"][0]
ancho_pred = r4["int_pred"][1] - r4["int_pred"][0]
assert ancho_pred > 10 * ancho_media, "con migliaia di dati l'IC della media è molto più stretto di quello di previsione"
assert en_rango(4.0) is True and en_rango(30.0) is False and en_rango(0.0) is False, "confronta con il minimo e il massimo di df['MedInc']"
print(f"Ampiezze: IC della media = {ancho_media:.3f}   intervallo di previsione = {ancho_pred:.3f}")
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
residuos = m_ej.resid.values
ajustados = m_ej.fittedvalues.values

techo = (df["MedHouseVal"] >= 5).values
n_techo = int(techo.sum())
res_techo = residuos[techo].mean()
res_resto = residuos[~techo].mean()
cook = m_ej.get_influence().cooks_distance[0]
pos_cook_max = int(np.argmax(cook))
corr_res_ajust = np.corrcoef(residuos, ajustados)[0, 1]

assert all(v is not None for v in (techo, n_techo, res_techo, res_resto, cook, pos_cook_max, corr_res_ajust)), "Completa le righe contrassegnate con TODO"
assert n_techo == int((df["MedHouseVal"] >= 5).sum()) and n_techo > 0, "conta i distretti con MedHouseVal >= 5"
assert np.isclose(res_techo, residuos[techo].mean()) and np.isclose(res_resto, residuos[~techo].mean()), "usa la maschera techo e la sua negazione"
assert res_techo > 0 > res_resto, "i censurati hanno residuo positivo: il loro valore reale (non troncato) sarebbe ancora maggiore"
assert len(cook) == len(df) and pos_cook_max == int(np.argmax(cook)), "pos_cook_max è la posizione del massimo di cook"
assert abs(corr_res_ajust) < 1e-8, "in OLS con intercetta, i residui non sono correlati con i valori adattati"
print(f"Censurati: {n_techo}   residuo medio censurati / altri: {res_techo:+.2f} / {res_resto:+.2f}   Cook max.: {cook.max():.3f}")
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
g = np.random.default_rng(11)
n_r = 3000
z_r = g.normal(0, 1, n_r)
x_r = 0.8 * z_r + g.normal(0, 1, n_r)
y_r = 0.3 * x_r + 1.0 * z_r + g.normal(0, 1, n_r)
datos_r = pd.DataFrame({"x": x_r, "y": y_r, "z": z_r})

m_c = smf.ols("y ~ x", data=datos_r).fit()
m_a = smf.ols("y ~ x + z", data=datos_r).fit()
coef_crudo, ee_crudo = m_c.params["x"], m_c.bse["x"]
coef_ajustado, ee_ajustado = m_a.params["x"], m_a.bse["x"]
sesgo = coef_crudo - coef_ajustado

assert all(v is not None for v in (coef_crudo, ee_crudo, coef_ajustado, ee_ajustado, sesgo)), "Completa le righe contrassegnate con TODO"
assert abs(coef_ajustado - 0.3) < 4 * ee_ajustado, "il modello con z deve recuperare l'effetto vero 0,3 (entro 4 errori standard)"
assert coef_crudo - 0.3 > 4 * ee_crudo, "il modello senza z sovrastima l'effetto (più di 4 errori standard)"
assert np.isclose(sesgo, coef_crudo - coef_ajustado) and sesgo > 0.3, "il bias è grezzo - corretto e qui supera 0,3"
print(f"Effetto vero: 0,3   grezzo: {coef_crudo:.3f} ± {ee_crudo:.3f}   corretto: {coef_ajustado:.3f} ± {ee_ajustado:.3f}")
print("bias:", round(sesgo, 3))
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (correlazione vs. causalità, lettura dell'output, R² corretto, intervalli di previsione, diagnostica dei residui)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*